# **Databricks data quality check notebook**

It checks:
- Silver tables exist and are not empty
- Gold tables exist and are not empty
- Null values in important columns
- Duplicate records
- Fact table business rules
- Dimension key integrity
- Sales and transactions validity
- Basic reconciliation between Silver and Gold

In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, count, sum as spark_sum

CATALOG = "grocery_catalog"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

silver_tables = {
    "train_clean": f"{CATALOG}.{SILVER_SCHEMA}.train_clean",
    "stores_clean": f"{CATALOG}.{SILVER_SCHEMA}.stores_clean",
    "transactions_clean": f"{CATALOG}.{SILVER_SCHEMA}.transactions_clean",
    "holidays_clean": f"{CATALOG}.{SILVER_SCHEMA}.holidays_clean",
    "oil_clean": f"{CATALOG}.{SILVER_SCHEMA}.oil_clean",
}

gold_tables = {
    "dim_date": f"{CATALOG}.{GOLD_SCHEMA}.dim_date",
    "dim_holiday": f"{CATALOG}.{GOLD_SCHEMA}.dim_holiday",
    "dim_oil": f"{CATALOG}.{GOLD_SCHEMA}.dim_oil",
    "dim_product": f"{CATALOG}.{GOLD_SCHEMA}.dim_product",
    "dim_promotion": f"{CATALOG}.{GOLD_SCHEMA}.dim_promotion",
    "dim_store": f"{CATALOG}.{GOLD_SCHEMA}.dim_store",
    "fact_sales": f"{CATALOG}.{GOLD_SCHEMA}.fact_sales",
}

def load_table(table_name: str):
    return spark.table(table_name)

def assert_table_not_empty(table_name: str):
    df = load_table(table_name)
    cnt = df.count()
    assert cnt > 0, f"FAILED: {table_name} is empty"
    print(f"PASSED: {table_name} has {cnt} records")

def assert_no_nulls(table_name: str, cols: list):
    df = load_table(table_name)
    for c in cols:
        null_cnt = df.filter(col(c).isNull()).count()
        assert null_cnt == 0, f"FAILED: {table_name}.{c} has {null_cnt} null values"
        print(f"PASSED: {table_name}.{c} has no nulls")

def assert_no_duplicates(table_name: str, key_cols: list):
    df = load_table(table_name)
    dup_cnt = (
        df.groupBy(key_cols)
          .count()
          .filter(col("count") > 1)
          .count()
    )
    assert dup_cnt == 0, f"FAILED: {table_name} has duplicate records on {key_cols}"
    print(f"PASSED: {table_name} has no duplicates on {key_cols}")

def assert_non_negative(table_name: str, cols: list):
    df = load_table(table_name)
    for c in cols:
        neg_cnt = df.filter(col(c) < 0).count()
        assert neg_cnt == 0, f"FAILED: {table_name}.{c} has {neg_cnt} negative values"
        print(f"PASSED: {table_name}.{c} has no negative values")

### **Checking all Silver and Gold tables are not empty**

In [0]:
for tbl in silver_tables.values():
    assert_table_not_empty(tbl)

for tbl in gold_tables.values():
    assert_table_not_empty(tbl)

PASSED: grocery_catalog.silver.train_clean has 3000888 records
PASSED: grocery_catalog.silver.stores_clean has 54 records
PASSED: grocery_catalog.silver.transactions_clean has 83488 records
PASSED: grocery_catalog.silver.holidays_clean has 350 records
PASSED: grocery_catalog.silver.oil_clean has 1218 records
PASSED: grocery_catalog.gold.dim_date has 1684 records
PASSED: grocery_catalog.gold.dim_holiday has 350 records
PASSED: grocery_catalog.gold.dim_oil has 1218 records
PASSED: grocery_catalog.gold.dim_product has 3000888 records
PASSED: grocery_catalog.gold.dim_promotion has 362 records
PASSED: grocery_catalog.gold.dim_store has 54 records
PASSED: grocery_catalog.gold.fact_sales has 3000888 records


### **Chechking Silver layer**

In [0]:
assert_no_nulls(silver_tables["train_clean"], ["date", "store_nbr", "family", "sales", "onpromotion"])
assert_no_duplicates(silver_tables["train_clean"], ["date", "store_nbr", "family"])
assert_non_negative(silver_tables["train_clean"], ["sales", "onpromotion"])

assert_no_nulls(silver_tables["stores_clean"], ["store_nbr", "city", "state", "type", "cluster"])
assert_no_duplicates(silver_tables["stores_clean"], ["store_nbr"])

assert_no_nulls(silver_tables["transactions_clean"], ["date", "store_nbr", "transactions"])
assert_no_duplicates(silver_tables["transactions_clean"], ["date", "store_nbr"])
assert_non_negative(silver_tables["transactions_clean"], ["transactions"])

assert_no_nulls(silver_tables["oil_clean"], ["date"])
assert_no_duplicates(silver_tables["oil_clean"], ["date"])

PASSED: grocery_catalog.silver.train_clean.date has no nulls
PASSED: grocery_catalog.silver.train_clean.store_nbr has no nulls
PASSED: grocery_catalog.silver.train_clean.family has no nulls
PASSED: grocery_catalog.silver.train_clean.sales has no nulls
PASSED: grocery_catalog.silver.train_clean.onpromotion has no nulls
PASSED: grocery_catalog.silver.train_clean has no duplicates on ['date', 'store_nbr', 'family']
PASSED: grocery_catalog.silver.train_clean.sales has no negative values
PASSED: grocery_catalog.silver.train_clean.onpromotion has no negative values
PASSED: grocery_catalog.silver.stores_clean.store_nbr has no nulls
PASSED: grocery_catalog.silver.stores_clean.city has no nulls
PASSED: grocery_catalog.silver.stores_clean.state has no nulls
PASSED: grocery_catalog.silver.stores_clean.type has no nulls
PASSED: grocery_catalog.silver.stores_clean.cluster has no nulls
PASSED: grocery_catalog.silver.stores_clean has no duplicates on ['store_nbr']
PASSED: grocery_catalog.silver.trans

### **Checking of Dimension tables**

In [0]:
assert_no_nulls(gold_tables["dim_date"], ["date", "year", "quarter", "month", "week", "day"])
assert_no_duplicates(gold_tables["dim_date"], ["date"])

assert_no_nulls(gold_tables["dim_store"], ["store_id", "city", "state", "type", "cluster"])
assert_no_duplicates(gold_tables["dim_store"], ["store_id"])

assert_no_nulls(gold_tables["dim_product"], ["product_id", "product_family"])
assert_no_duplicates(gold_tables["dim_product"], ["product_id"])

assert_no_nulls(gold_tables["dim_promotion"], ["onpromotion"])
assert_no_duplicates(gold_tables["dim_promotion"], ["onpromotion"])

assert_no_nulls(gold_tables["dim_oil"], ["date"])
assert_no_duplicates(gold_tables["dim_oil"], ["date"])

assert_no_nulls(gold_tables["dim_holiday"], ["date"])

PASSED: grocery_catalog.gold.dim_date.date has no nulls
PASSED: grocery_catalog.gold.dim_date.year has no nulls
PASSED: grocery_catalog.gold.dim_date.quarter has no nulls
PASSED: grocery_catalog.gold.dim_date.month has no nulls
PASSED: grocery_catalog.gold.dim_date.week has no nulls
PASSED: grocery_catalog.gold.dim_date.day has no nulls
PASSED: grocery_catalog.gold.dim_date has no duplicates on ['date']
PASSED: grocery_catalog.gold.dim_store.store_id has no nulls
PASSED: grocery_catalog.gold.dim_store.city has no nulls
PASSED: grocery_catalog.gold.dim_store.state has no nulls
PASSED: grocery_catalog.gold.dim_store.type has no nulls
PASSED: grocery_catalog.gold.dim_store.cluster has no nulls
PASSED: grocery_catalog.gold.dim_store has no duplicates on ['store_id']
PASSED: grocery_catalog.gold.dim_product.product_id has no nulls
PASSED: grocery_catalog.gold.dim_product.product_family has no nulls
PASSED: grocery_catalog.gold.dim_product has no duplicates on ['product_id']
PASSED: grocery_

### **Checking of Fact Table**

In [0]:
assert_no_nulls(gold_tables["fact_sales"], ["date", "store_nbr", "family", "sales", "onpromotion"])
assert_no_duplicates(gold_tables["fact_sales"], ["date", "store_nbr", "family"])
assert_non_negative(gold_tables["fact_sales"], ["sales", "onpromotion", "transactions"])

PASSED: grocery_catalog.gold.fact_sales.date has no nulls
PASSED: grocery_catalog.gold.fact_sales.store_nbr has no nulls
PASSED: grocery_catalog.gold.fact_sales.family has no nulls
PASSED: grocery_catalog.gold.fact_sales.sales has no nulls
PASSED: grocery_catalog.gold.fact_sales.onpromotion has no nulls
PASSED: grocery_catalog.gold.fact_sales has no duplicates on ['date', 'store_nbr', 'family']
PASSED: grocery_catalog.gold.fact_sales.sales has no negative values
PASSED: grocery_catalog.gold.fact_sales.onpromotion has no negative values
PASSED: grocery_catalog.gold.fact_sales.transactions has no negative values


### **Checking of Refrential Integrity**

In [0]:
fact_sales = load_table(gold_tables["fact_sales"])
dim_date = load_table(gold_tables["dim_date"])
dim_store = load_table(gold_tables["dim_store"])
dim_product = load_table(gold_tables["dim_product"])
dim_promotion = load_table(gold_tables["dim_promotion"])

bad_date = fact_sales.join(dim_date, fact_sales.date == dim_date.date, "left_anti").count()
assert bad_date == 0, f"FAILED: fact_sales has {bad_date} records with invalid date"
print("PASSED: fact_sales date integrity check")

bad_store = fact_sales.join(dim_store, fact_sales.store_nbr == dim_store.store_id, "left_anti").count()
assert bad_store == 0, f"FAILED: fact_sales has {bad_store} records with invalid store"
print("PASSED: fact_sales store integrity check")

bad_product = fact_sales.join(dim_product, fact_sales.family == dim_product.product_family, "left_anti").count()
assert bad_product == 0, f"FAILED: fact_sales has {bad_product} records with invalid product"
print("PASSED: fact_sales product integrity check")

bad_promo = fact_sales.join(dim_promotion, fact_sales.onpromotion == dim_promotion.onpromotion, "left_anti").count()
assert bad_promo == 0, f"FAILED: fact_sales has {bad_promo} records with invalid promotion value"
print("PASSED: fact_sales promotion integrity check")

PASSED: fact_sales date integrity check
PASSED: fact_sales store integrity check
PASSED: fact_sales product integrity check
PASSED: fact_sales promotion integrity check


### **Reconciliation checks between Silver and Gold**

Reconciliation checks between Silver and Gold

In [0]:
train_clean = load_table(silver_tables["train_clean"])

silver_count = train_clean.count()
gold_count = fact_sales.count()

assert silver_count == gold_count, f"FAILED: train_clean count ({silver_count}) != fact_sales count ({gold_count})"
print("PASSED: fact_sales row count matches train_clean")

PASSED: fact_sales row count matches train_clean


Total sales in Silver and Gold should match

In [0]:
silver_sales_total = train_clean.agg(spark_sum("sales").alias("total")).collect()[0]["total"]
gold_sales_total = fact_sales.agg(spark_sum("sales").alias("total")).collect()[0]["total"]

assert round(silver_sales_total, 2) == round(gold_sales_total, 2), \
    f"FAILED: total sales mismatch silver={silver_sales_total}, gold={gold_sales_total}"

print("PASSED: total sales match between Silver and Gold")

PASSED: total sales match between Silver and Gold


In [0]:
print("ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY")


ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY


# **By Using Pytest**

In [0]:
test_code = '''
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

CATALOG = "grocery_catalog"
GOLD_SCHEMA = "gold"

@pytest.fixture(scope="session")
def spark():
    spark = SparkSession.builder.appName("gold-layer-pytest").getOrCreate()
    yield spark

def test_fact_sales_not_empty(spark):
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
    assert df.count() > 0, "fact_sales is empty"

def test_fact_sales_no_null_sales(spark):
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
    null_count = df.filter(col("sales").isNull()).count()
    assert null_count == 0, f"sales has {null_count} null values"

def test_fact_sales_no_duplicates(spark):
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
    dup_count = (
        df.groupBy("date", "store_nbr", "family")
          .count()
          .filter(col("count") > 1)
          .count()
    )
    assert dup_count == 0, f"Duplicate fact rows found: {dup_count}"

def test_fact_sales_non_negative_sales(spark):
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
    neg_count = df.filter(col("sales") < 0).count()
    assert neg_count == 0, f"Negative sales found: {neg_count}"

def test_dim_store_unique(spark):
    df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_store")
    dup_count = (
        df.groupBy("store_id")
          .count()
          .filter(col("count") > 1)
          .count()
    )
    assert dup_count == 0, f"Duplicate store_id found: {dup_count}"

def test_fact_store_integrity(spark):
    fact = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
    dim = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_store")

    invalid = fact.join(
        dim,
        fact.store_nbr == dim.store_id,
        "left_anti"
    ).count()

    assert invalid == 0, f"Invalid store references in fact_sales: {invalid}"

def test_sales_reconciliation(spark):
    fact = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_sales")
    total_sales = fact.agg(spark_sum("sales").alias("total_sales")).collect()[0]["total_sales"]
    assert total_sales is not None and total_sales > 0, "Total sales reconciliation failed"

def test_sales_trend_available(spark):
    df = spark.table("grocery_catalog.gold.fact_sales")
    result = df.groupBy("date").agg(spark_sum("sales").alias("total_sales"))
    assert result.count() > 0, "Sales trend result is empty"


def test_product_demand_available(spark):
    df = spark.table("grocery_catalog.gold.fact_sales")
    result = df.groupBy("family").agg(spark_sum("sales").alias("total_sales"))
    assert result.count() > 0, "Product demand result is empty"

def test_store_performance_available(spark):
    df = spark.table("grocery_catalog.gold.fact_sales")
    result = df.groupBy("store_nbr").agg(spark_sum("sales").alias("total_sales"))
    assert result.count() > 0, "Store performance result is empty"
'''
with open("/tmp/test_gold_layer.py", "w") as f:
    f.write(test_code)

print("pytest file created: /tmp/test_gold_layer.py")


pytest file created: /tmp/test_gold_layer.py


In [0]:
import pytest

result = pytest.main(["-q", "/tmp/test_gold_layer.py"])
print("Pytest exit code:", result)

assert result == 0, "Some pytest tests failed"

.......                                                                  [100%]
7 passed in 2.42s
Pytest exit code: 0
